# Data Analysis: Generate Table from Data/20251016

This notebook reads all data files from the Data/20251016 folder and generates a consolidated table with the information.

## 1. Import Required Libraries

Import pandas for data manipulation, os and glob for file handling, and pathlib for path operations.

In [ ]:
import pandas as pd
import os
import glob
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")

## 2. Set Data Directory Path

Define the path to the Data/20251016 folder and verify it exists using os.path or pathlib.

In [ ]:
# Define the data directory path
data_dir = Path("Data/20251016")

# Check if directory exists
if data_dir.exists():
    print(f"✓ Directory found: {data_dir.absolute()}")
    print(f"  Directory is {'absolute' if data_dir.is_absolute() else 'relative'}")
else:
    print(f"✗ Directory not found: {data_dir}")
    print("  Creating directory...")
    data_dir.mkdir(parents=True, exist_ok=True)
    print(f"  Directory created at: {data_dir.absolute()}")

## 3. List Files in Directory

Use glob or os.listdir to get all data files in the directory, filtering by file extension if needed (e.g., .csv, .xlsx, .json).

In [ ]:
# Get all files in the directory
all_files = list(data_dir.glob("*"))

# Separate files by type
csv_files = list(data_dir.glob("*.csv"))
excel_files = list(data_dir.glob("*.xlsx")) + list(data_dir.glob("*.xls"))
json_files = list(data_dir.glob("*.json"))
txt_files = list(data_dir.glob("*.txt"))

# Display file summary
print(f"Total files found: {len(all_files)}")
print(f"├── CSV files: {len(csv_files)}")
print(f"├── Excel files: {len(excel_files)}")
print(f"├── JSON files: {len(json_files)}")
print(f"└── Text files: {len(txt_files)}")

print("\nDetailed file list:")
for file in all_files:
    if file.is_file():
        size_kb = file.stat().st_size / 1024
        print(f"  - {file.name} ({size_kb:.2f} KB)")

## 4. Read and Process Data Files

Loop through each file in the directory and read them into pandas DataFrames based on their file type, handling different formats appropriately.

In [ ]:
# Function to read different file types
def read_file_to_dataframe(file_path):
    """
    Read a file into a pandas DataFrame based on its extension.
    """
    file_path = Path(file_path)
    extension = file_path.suffix.lower()
    
    try:
        if extension == '.csv':
            return pd.read_csv(file_path)
        elif extension in ['.xlsx', '.xls']:
            return pd.read_excel(file_path)
        elif extension == '.json':
            # Try different JSON orientations
            try:
                return pd.read_json(file_path)
            except:
                with open(file_path, 'r') as f:
                    data = json.load(f)
                    if isinstance(data, list):
                        return pd.DataFrame(data)
                    else:
                        return pd.DataFrame([data])
        elif extension == '.txt':
            # Try to read as CSV with various delimiters
            for delimiter in ['\t', ',', '|', ';']:
                try:
                    df = pd.read_csv(file_path, delimiter=delimiter)
                    if len(df.columns) > 1:
                        return df
                except:
                    continue
            # If all else fails, read as single column
            return pd.read_csv(file_path, header=None, names=['content'])
        else:
            print(f"  ⚠ Unsupported file type: {extension}")
            return None
    except Exception as e:
        print(f"  ✗ Error reading {file_path.name}: {str(e)}")
        return None

# Read all data files
dataframes = {}
for file in all_files:
    if file.is_file():
        print(f"Reading {file.name}...")
        df = read_file_to_dataframe(file)
        if df is not None:
            dataframes[file.name] = df
            print(f"  ✓ Loaded {len(df)} rows × {len(df.columns)} columns")
        else:
            print(f"  ✗ Could not load {file.name}")

print(f"\nSuccessfully loaded {len(dataframes)} file(s)")

## 5. Create Consolidated Table

Combine all loaded data into a single table using pandas concat or merge operations, adding a source filename column if needed.

In [ ]:
# Create consolidated table
if dataframes:
    # Add source filename column to each dataframe
    for filename, df in dataframes.items():
        df['source_file'] = filename
    
    # Concatenate all dataframes
    try:
        consolidated_df = pd.concat(dataframes.values(), ignore_index=True, sort=False)
        print(f"✓ Consolidated table created successfully!")
        print(f"  Total rows: {len(consolidated_df)}")
        print(f"  Total columns: {len(consolidated_df.columns)}")
        
        # Display basic statistics
        print(f"\nColumn Information:")
        print("-" * 40)
        for col in consolidated_df.columns:
            non_null = consolidated_df[col].notna().sum()
            dtype = consolidated_df[col].dtype
            print(f"  {col}: {dtype} ({non_null} non-null values)")
            
    except Exception as e:
        print(f"✗ Error creating consolidated table: {str(e)}")
        # Try alternative approach: create summary table
        print("\nCreating summary table instead...")
        summary_data = []
        for filename, df in dataframes.items():
            summary_data.append({
                'filename': filename,
                'rows': len(df),
                'columns': len(df.columns),
                'column_names': ', '.join(df.columns.tolist()[:5]),
                'memory_usage': f"{df.memory_usage().sum() / 1024:.2f} KB"
            })
        consolidated_df = pd.DataFrame(summary_data)
        print("✓ Summary table created")
else:
    print("No data files found or loaded. Creating empty DataFrame.")
    consolidated_df = pd.DataFrame()

## 6. Display and Export Table

Display the final table using pandas display methods and export it to a desired format (CSV, Excel, etc.).

In [ ]:
# Display the consolidated table
if not consolidated_df.empty:
    print("First 10 rows of the consolidated table:")
    print("=" * 80)
    display(consolidated_df.head(10))
    
    print(f"\nTable shape: {consolidated_df.shape}")
    print(f"Memory usage: {consolidated_df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")
    
    # Display basic statistics for numeric columns
    numeric_cols = consolidated_df.select_dtypes(include=['number']).columns
    if len(numeric_cols) > 0:
        print("\nNumeric column statistics:")
        print("=" * 80)
        display(consolidated_df[numeric_cols].describe())
else:
    print("No data to display.")

In [ ]:
# Export the consolidated table
if not consolidated_df.empty:
    # Define output directory
    output_dir = Path("output")
    output_dir.mkdir(exist_ok=True)
    
    # Export to CSV
    csv_output = output_dir / "consolidated_data_20251016.csv"
    consolidated_df.to_csv(csv_output, index=False)
    print(f"✓ Exported to CSV: {csv_output}")
    
    # Export to Excel with formatting
    excel_output = output_dir / "consolidated_data_20251016.xlsx"
    with pd.ExcelWriter(excel_output, engine='openpyxl') as writer:
        consolidated_df.to_excel(writer, sheet_name='Data', index=False)
        
        # Add summary sheet if we have multiple source files
        if 'source_file' in consolidated_df.columns:
            summary = consolidated_df.groupby('source_file').agg({
                consolidated_df.columns[0]: 'count'
            }).rename(columns={consolidated_df.columns[0]: 'row_count'})
            summary.to_excel(writer, sheet_name='Summary')
    
    print(f"✓ Exported to Excel: {excel_output}")
    
    # Export to JSON (records format)
    json_output = output_dir / "consolidated_data_20251016.json"
    consolidated_df.to_json(json_output, orient='records', indent=2)
    print(f"✓ Exported to JSON: {json_output}")
    
    print(f"\nAll exports completed successfully in '{output_dir}' directory!")
else:
    print("No data to export.")

In [ ]:
# Optional: Create a quick visualization if applicable
if not consolidated_df.empty:
    try:
        import matplotlib.pyplot as plt
        
        # Create a simple visualization based on available data
        fig, ax = plt.subplots(figsize=(10, 6))
        
        if 'source_file' in consolidated_df.columns:
            # Plot file distribution
            file_counts = consolidated_df['source_file'].value_counts()
            file_counts.plot(kind='bar', ax=ax)
            ax.set_title('Records per Source File')
            ax.set_xlabel('Source File')
            ax.set_ylabel('Number of Records')
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()
        else:
            print("No suitable columns for visualization")
            
    except ImportError:
        print("Matplotlib not available for visualization")